In [ ]:
# Setup
import sys
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(BASE_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.config import PROCESSED_DIR, REPORTS_DIR, MODELS_DIR

import pickle
import json

COMMODITIES = ['corn', 'soybeans', 'wheat']

print(f"Módulos cargados correctamente")
print(f"Directorio base: {BASE_DIR}")

## 1. Cargar Predicciones de Modelos

Usamos predicciones del mejor modelo (determinado por walk-forward validation).

In [ ]:
# Load dataset
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'base_tp3_trabajo.csv'))
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Dataset: {df.shape[0]} obs, período {df['date'].min()} → {df['date'].max()}")

# Train/test split (70-30)
train_size = int(len(df) * 0.7)
df_test = df.iloc[train_size:].copy()

print(f"\nTest set para backtesting:")
print(f"  Observaciones: {len(df_test)}")
print(f"  Período: {df_test['date'].min()} → {df_test['date'].max()}")
print(f"  Años: {(df_test['date'].max() - df_test['date'].min()).days / 365.25:.1f}")

In [ ]:
# Check available model results
model_files = {
    'baseline': 'baseline_models_results.json',
    'tree': 'tree_models_results.json',
    'lstm': 'lstm_multivariate_results.json',
    'vmd_lstm': 'vmd_lstm_results.json',
    'walk_forward': 'walk_forward_validation_results.json',
    'ensemble': 'ensemble_arima_lstm_results.json'
}

available_models = {}
for name, file in model_files.items():
    path = os.path.join(PROCESSED_DIR, file)
    if os.path.exists(path):
        with open(path, 'r') as f:
            available_models[name] = json.load(f)
        print(f"✓ {name}: {file}")
    else:
        print(f"✗ {name}: No disponible")

# Select best model based on walk-forward R² and DA
# TODO: Update this based on actual walk-forward LSTM results
best_model_name = 'lstm'  # Placeholder
print(f"\n→ Modelo seleccionado para trading: {best_model_name.upper()}")

## 2. Generar Predicciones en Test Set

Cargamos modelo entrenado y generamos predicciones para test period.

In [ ]:
# Function to load model and generate predictions
def generate_predictions(commodity, model_type='lstm'):
    """Load model and generate test predictions"""
    
    if model_type == 'lstm':
        # Load LSTM model
        from tensorflow import keras
        model_path = os.path.join(MODELS_DIR, f'lstm_multivariate_{commodity}.h5')
        model = keras.models.load_model(model_path)
        
        # Load scalers
        with open(os.path.join(MODELS_DIR, 'multivariate_lstm_scalers.pkl'), 'rb') as f:
            scalers = pickle.load(f)
        
        scaler_X = scalers[commodity]['X']
        scaler_y = scalers[commodity]['y']
        
        # Prepare features
        exclude_cols = ['date'] + [c for c in df.columns if '_target' in c or '_spot' in c]
        feature_cols = [c for c in df.columns if c not in exclude_cols]
        
        X_test = df_test[feature_cols].values
        X_test_scaled = scaler_X.transform(X_test)
        
        # Create sequences
        seq_len = 30
        X_seq = []
        for i in range(seq_len, len(X_test_scaled)):
            X_seq.append(X_test_scaled[i-seq_len:i])
        X_seq = np.array(X_seq)
        
        # Predict
        y_pred_scaled = model.predict(X_seq, verbose=0)
        y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
        
        # Align with dates (skip first seq_len)
        dates_aligned = df_test['date'].iloc[seq_len:].values
        spot_aligned = df_test[f"{commodity}_spot"].iloc[seq_len:].values
        target_aligned = df_test[f"{commodity}_target"].iloc[seq_len:].values
        
        return pd.DataFrame({
            'date': dates_aligned,
            'spot_price': spot_aligned,
            'target_price': target_aligned,
            'predicted_price': y_pred
        })
    
    elif model_type == 'baseline':
        # Load baseline model (Lasso/Ridge)
        with open(os.path.join(MODELS_DIR, 'baseline_models.pkl'), 'rb') as f:
            models = pickle.load(f)
        
        # TODO: Implement baseline prediction logic
        pass
    
    else:
        raise ValueError(f"Model type {model_type} not implemented")

# Generate predictions for all commodities
predictions = {}
for commodity in COMMODITIES:
    print(f"Generando predicciones - {commodity.upper()}...")
    predictions[commodity] = generate_predictions(commodity, model_type=best_model_name)
    print(f"  → {len(predictions[commodity])} predicciones")

# Display sample
print("\nMuestra de predicciones (Corn):")
print(predictions['corn'].head())

## 3. Implementar Estrategia de Trading

In [ ]:
class TradingStrategy:
    """Backtesting de estrategia basada en predicciones"""
    
    def __init__(self, predictions_df, initial_capital=100000, 
                 commission=0.002, slippage=0.0005, threshold=0.01):
        """
        Args:
            predictions_df: DataFrame con date, spot_price, target_price, predicted_price
            initial_capital: Capital inicial en USD
            commission: % comisión por operación (default 0.2%)
            slippage: % slippage bid-ask (default 0.05%)
            threshold: % diferencia mínima pred vs spot para generar señal (default 1%)
        """
        self.df = predictions_df.copy()
        self.initial_capital = initial_capital
        self.commission = commission
        self.slippage = slippage
        self.threshold = threshold
        
        # State variables
        self.capital = initial_capital
        self.position = 0  # Current position (contracts)
        self.trades = []
        self.equity_curve = []
        
    def generate_signals(self):
        """Generar señales de compra/venta"""
        signals = []
        
        for i, row in self.df.iterrows():
            spot = row['spot_price']
            pred = row['predicted_price']
            
            # Expected return
            expected_return = (pred - spot) / spot
            
            if expected_return > self.threshold:
                signal = 'BUY'  # Expect price increase
            elif expected_return < -self.threshold:
                signal = 'SELL'  # Expect price decrease
            else:
                signal = 'HOLD'  # No clear signal
            
            signals.append(signal)
        
        self.df['signal'] = signals
        return self.df
    
    def backtest(self):
        """Execute backtest"""
        self.generate_signals()
        
        for i, row in self.df.iterrows():
            date = row['date']
            spot = row['spot_price']
            target = row['target_price']  # Actual future price (h=7)
            signal = row['signal']
            
            # Calculate realized return (from spot to target)
            realized_return = (target - spot) / spot
            
            # Execute trade based on signal
            if signal == 'BUY' and self.position == 0:
                # Enter long position
                contracts = self.capital / (spot * (1 + self.commission + self.slippage))
                entry_cost = contracts * spot * (1 + self.commission + self.slippage)
                self.position = contracts
                self.capital -= entry_cost
                
                self.trades.append({
                    'date': date,
                    'type': 'LONG_ENTRY',
                    'price': spot,
                    'contracts': contracts,
                    'cost': entry_cost
                })
            
            elif signal == 'SELL' and self.position == 0:
                # Enter short position (simplified: assume can short)
                contracts = self.capital / (spot * (1 + self.commission + self.slippage))
                self.position = -contracts
                self.capital += contracts * spot * (1 - self.commission - self.slippage)
                
                self.trades.append({
                    'date': date,
                    'type': 'SHORT_ENTRY',
                    'price': spot,
                    'contracts': contracts,
                    'cost': 0
                })
            
            elif signal == 'HOLD' and self.position != 0:
                # Close position (return to neutral)
                if self.position > 0:  # Close long
                    exit_value = self.position * target * (1 - self.commission - self.slippage)
                    self.capital += exit_value
                    pnl = exit_value - (self.position * spot)
                    
                    self.trades.append({
                        'date': date,
                        'type': 'LONG_EXIT',
                        'price': target,
                        'contracts': self.position,
                        'pnl': pnl
                    })
                else:  # Close short
                    exit_cost = abs(self.position) * target * (1 + self.commission + self.slippage)
                    self.capital -= exit_cost
                    pnl = (abs(self.position) * spot) - exit_cost
                    
                    self.trades.append({
                        'date': date,
                        'type': 'SHORT_EXIT',
                        'price': target,
                        'contracts': abs(self.position),
                        'pnl': pnl
                    })
                
                self.position = 0
            
            # Update equity curve
            if self.position > 0:
                equity = self.capital + (self.position * target)
            elif self.position < 0:
                equity = self.capital - (abs(self.position) * target)
            else:
                equity = self.capital
            
            self.equity_curve.append({
                'date': date,
                'equity': equity,
                'position': self.position
            })
        
        # Close any remaining position at end
        if self.position != 0:
            last_price = self.df.iloc[-1]['target_price']
            if self.position > 0:
                self.capital += self.position * last_price * (1 - self.commission)
            else:
                self.capital -= abs(self.position) * last_price * (1 + self.commission)
            self.position = 0
        
        return pd.DataFrame(self.trades), pd.DataFrame(self.equity_curve)
    
    def calculate_metrics(self, equity_df):
        """Calculate performance metrics"""
        final_equity = equity_df['equity'].iloc[-1]
        total_return = (final_equity - self.initial_capital) / self.initial_capital * 100
        
        # Calculate returns
        equity_df['returns'] = equity_df['equity'].pct_change()
        
        # Sharpe Ratio (annualized, assuming 252 trading days)
        mean_return = equity_df['returns'].mean()
        std_return = equity_df['returns'].std()
        sharpe = (mean_return / std_return) * np.sqrt(252) if std_return > 0 else 0
        
        # Max Drawdown
        cummax = equity_df['equity'].cummax()
        drawdown = (equity_df['equity'] - cummax) / cummax
        max_drawdown = drawdown.min() * 100
        
        # Win rate
        trades_df = pd.DataFrame(self.trades)
        if 'pnl' in trades_df.columns:
            winning_trades = (trades_df['pnl'] > 0).sum()
            total_trades = trades_df['pnl'].notna().sum()
            win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
            
            # Profit factor
            gross_profit = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
            gross_loss = abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
            profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else np.inf
        else:
            win_rate = 0
            profit_factor = 0
        
        return {
            'total_return_pct': total_return,
            'sharpe_ratio': sharpe,
            'max_drawdown_pct': max_drawdown,
            'win_rate_pct': win_rate,
            'profit_factor': profit_factor,
            'total_trades': len(self.trades),
            'final_equity': final_equity
        }

print("✓ Clase TradingStrategy definida")

## 4. Ejecutar Backtesting

In [ ]:
# Backtest each commodity
results = []

for commodity in COMMODITIES:
    print(f"\n{'='*60}")
    print(f"Backtesting - {commodity.upper()}")
    print(f"{'='*60}")
    
    # Initialize strategy
    strategy = TradingStrategy(
        predictions[commodity],
        initial_capital=100000,
        commission=0.002,  # 0.2%
        slippage=0.0005,   # 0.05%
        threshold=0.01     # 1% signal threshold
    )
    
    # Run backtest
    trades_df, equity_df = strategy.backtest()
    
    # Calculate metrics
    metrics = strategy.calculate_metrics(equity_df)
    
    print(f"\nResultados:")
    print(f"  Total Return: {metrics['total_return_pct']:.2f}%")
    print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.3f}")
    print(f"  Max Drawdown: {metrics['max_drawdown_pct']:.2f}%")
    print(f"  Win Rate: {metrics['win_rate_pct']:.2f}%")
    print(f"  Profit Factor: {metrics['profit_factor']:.3f}")
    print(f"  Total Trades: {metrics['total_trades']}")
    print(f"  Final Equity: ${metrics['final_equity']:,.2f}")
    
    # Save results
    results.append({
        'commodity': commodity,
        **metrics
    })
    
    # Save detailed data
    trades_df.to_csv(os.path.join(PROCESSED_DIR, f'trades_{commodity}.csv'), index=False)
    equity_df.to_csv(os.path.join(PROCESSED_DIR, f'equity_curve_{commodity}.csv'), index=False)

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("RESUMEN COMPARATIVO - TRADING PERFORMANCE")
print("="*80)
print(results_df.to_string(index=False))

## 5. Visualización de Equity Curves

In [ ]:
# Plot equity curves
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for i, commodity in enumerate(COMMODITIES):
    # Load equity curve
    equity_df = pd.read_csv(os.path.join(PROCESSED_DIR, f'equity_curve_{commodity}.csv'))
    equity_df['date'] = pd.to_datetime(equity_df['date'])
    
    ax = axes[i]
    ax.plot(equity_df['date'], equity_df['equity'], linewidth=2, label='Portfolio Value')
    ax.axhline(100000, color='gray', linestyle='--', linewidth=1, label='Initial Capital')
    
    # Shade drawdown periods
    cummax = equity_df['equity'].cummax()
    drawdown = equity_df['equity'] < cummax
    ax.fill_between(equity_df['date'], 0, equity_df['equity'].max() * 1.1, 
                     where=drawdown, alpha=0.2, color='red', label='Drawdown Periods')
    
    ax.set_title(f"{commodity.upper()} - Equity Curve", fontsize=13, fontweight='bold')
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Portfolio Value (USD)', fontsize=11)
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    ax.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'figures', 'trading_equity_curves.png'), 
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: trading_equity_curves.png")

## 6. Comparación de Métricas

In [ ]:
# Bar plots: Comparative metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = [
    ('total_return_pct', 'Total Return (%)'),
    ('sharpe_ratio', 'Sharpe Ratio'),
    ('max_drawdown_pct', 'Max Drawdown (%)'),
    ('win_rate_pct', 'Win Rate (%)')
]

for idx, (metric, title) in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    values = results_df[metric].values
    colors = ['green' if v > 0 else 'red' for v in values]
    
    ax.bar(results_df['commodity'].str.capitalize(), values, color=colors, alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(title, fontsize=11)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(values):
        ax.text(i, v, f'{v:.2f}', ha='center', va='bottom' if v > 0 else 'top', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'figures', 'trading_metrics_comparison.png'), 
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado: trading_metrics_comparison.png")

## 7. Guardar Resultados

In [ ]:
# Save summary
output = {
    'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_used': best_model_name,
    'parameters': {
        'initial_capital': 100000,
        'commission': 0.002,
        'slippage': 0.0005,
        'threshold': 0.01
    },
    'results': results_df.to_dict(orient='records')
}

with open(os.path.join(PROCESSED_DIR, 'trading_simulation_results.json'), 'w') as f:
    json.dump(output, f, indent=2)

results_df.to_csv(os.path.join(PROCESSED_DIR, 'trading_simulation_summary.csv'), index=False)

print("✓ Resultados guardados:")
print(f"  - trading_simulation_results.json")
print(f"  - trading_simulation_summary.csv")
print(f"  - trades_{{commodity}}.csv (3 archivos)")
print(f"  - equity_curve_{{commodity}}.csv (3 archivos)")

## 8. Conclusiones

**Criterios de viabilidad:**
- **Sharpe Ratio > 1.0:** Estrategia viable (mejor que benchmark)
- **Max Drawdown < 20%:** Riesgo aceptable
- **Win Rate > 55%:** Predictive power superior a aleatorio
- **Profit Factor > 1.5:** Ganancias superan pérdidas con margen

**Si métricas NO cumplen:**
- Revisar threshold de señales (más conservador)
- Ajustar tamaño de posiciones (risk management)
- Considerar estrategias alternativas (mean reversion, pairs trading)
- Modelo predictivo puede no ser adecuado para trading real